# NoveltyBench distinct-10 for the EGRA story methods

How many **genuinely different** stories each method writes, judged by the
functional-equivalence classifier from NoveltyBench (Zhang et al., 2025): a
DeBERTa-v3-large model fine-tuned on human judgements of whether two outputs are
worth seeing separately. Rewording the same story does not count as a new one.

* Two stories are the same when the judge scores the pair above **0.102** (the
  benchmark's cut-off); each story sees the first 128 tokens, as in the benchmark.
* A story joins the first class whose head it matches, otherwise it starts a new one.
* **distinct-10** is the number of classes among 10 stories, averaged over 500
  random subsets of each method's coherent stories. Higher is more varied.
* **same-story pairs** is the share of all pairs the judge calls the same story.
  Lower is more varied.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**, then
Runtime -> Run all. The stories (every coherent story from each method) are
downloaded from the project repository; results are saved to
`My Drive/EGRA creativity metrics/` as `novelty_results.json` and
`novelty_results.csv`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, urllib.request
FOLDER = '/content/drive/MyDrive/EGRA creativity metrics'   # results are saved here
os.makedirs(FOLDER, exist_ok=True)
URL = 'https://raw.githubusercontent.com/haziq-exe/NoiseEGRA/middle-school-register/eval/coherent_stories.json'
DATA = '/content/coherent_stories.json'
urllib.request.urlretrieve(URL, DATA)
print('stories downloaded:', os.path.getsize(DATA), 'bytes; results will go to', FOLDER)


In [ ]:
!pip -q install sentencepiece protobuf
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none -- switch the runtime to a GPU')


In [ ]:
import json, random, time
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
THRESHOLD = 0.102      # NoveltyBench: two outputs are the same when the judge scores above this
K, SUBSETS = 10, 500   # distinct-10, averaged over random subsets of 10 stories

tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
judge = AutoModelForSequenceClassification.from_pretrained(
    "yimingzhang/deberta-v3-large-generation-similarity").to(DEVICE).eval()
if DEVICE == "cuda":
    judge = judge.half()
_enc = {}


def enc(s):
    if s not in _enc:
        _enc[s] = tok.encode(s, truncation=True, max_length=128, add_special_tokens=False)
    return _enc[s]


def pair_scores(pairs, batch=64):
    """The judge's probability that each pair is the same story, as NoveltyBench builds its input."""
    out = []
    for i in range(0, len(pairs), batch):
        ids, tts = [], []
        for a, b in pairs[i:i + batch]:
            x = [tok.cls_token_id] + enc(a) + [tok.sep_token_id]
            first = len(x)
            x += enc(b) + [tok.sep_token_id]
            ids.append(x)
            tts.append([0] * first + [1] * (len(x) - first))
        n = max(len(x) for x in ids)
        att = [[1] * len(x) + [0] * (n - len(x)) for x in ids]
        ids = [x + [tok.pad_token_id] * (n - len(x)) for x in ids]
        tts = [t + [0] * (n - len(t)) for t in tts]
        with torch.inference_mode():
            logits = judge(input_ids=torch.tensor(ids, device=DEVICE),
                           token_type_ids=torch.tensor(tts, device=DEVICE),
                           attention_mask=torch.tensor(att, device=DEVICE)).logits
        out += logits.float().softmax(-1)[:, 1].cpu().tolist()
    return out


def distinct_k(texts, k=None, subsets=None, seed=0):
    """Distinct stories among k, averaged over random subsets, with a 95% range."""
    n = len(texts)
    k = min(k or K, n)
    subsets = subsets or SUBSETS
    idx = [(i, j) for i in range(n) for j in range(i)]
    scores = pair_scores([(texts[i], texts[j]) for i, j in idx])
    same = {}
    for (i, j), s in zip(idx, scores):
        same[(i, j)] = same[(j, i)] = s > THRESHOLD
    rng = random.Random(seed)
    counts = []
    for _ in range(subsets):
        heads = []
        for i in rng.sample(range(n), k):
            if not any(same[(i, h)] for h in heads):   # joins the first class it matches, as NoveltyBench does
                heads.append(i)
        counts.append(len(heads))
    counts.sort()
    return dict(distinct=sum(counts) / len(counts),
                low=counts[int(0.025 * subsets)], high=counts[int(0.975 * subsets) - 1],
                same_story_pairs=sum(s > THRESHOLD for s in scores) / len(scores), stories=n)


def score_all(data, log=print):
    results = {}
    for prompt, methods in data.items():
        results[prompt] = {}
        for name, texts in methods.items():
            t0 = time.time()
            r = distinct_k(texts)
            results[prompt][name] = r
            log(f"{prompt:6s} {name:26s} distinct-10 {r['distinct']:.2f} "
                f"(95% of subsets {r['low']}-{r['high']})  same-story pairs "
                f"{r['same_story_pairs']:.1%}  {r['stories']} stories  {time.time() - t0:.0f}s")
    return results


In [ ]:
import json, csv
data = json.load(open(DATA))
print({p: {m: len(t) for m, t in ms.items()} for p, ms in data.items()})
results = score_all(data)
json.dump(results, open(os.path.join(FOLDER, 'novelty_results.json'), 'w'), indent=1)
with open(os.path.join(FOLDER, 'novelty_results.csv'), 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['prompt', 'method', 'distinct_10', 'subsets_low', 'subsets_high', 'same_story_pairs', 'stories'])
    for p, ms in results.items():
        for m, r in ms.items():
            w.writerow([p, m, round(r['distinct'], 2), r['low'], r['high'], round(r['same_story_pairs'], 4), r['stories']])
print('saved to', FOLDER)
